# <font color="#418FDE" size="6.5" uppercase>**Performance Metrics**</font>

>Last update: 20260608.
    
By the end of this Lecture, you will be able to:
- Compute descriptive measures for production and process performance data. 
- Calculate utilization, throughput, yield, and downtime metrics in Python. 
- Interpret metric results in relation to process improvement decisions. 


## **1. Descriptive Measures**

### **1.1. Mean Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Industrial Engineers/Module_04/Lecture_A/image_01_01.jpg?v=1780942185" width="250">



>* Means summarize typical process performance
>* Averages support planning and comparisons

>* Averages work best with consistent data
>* Extreme values require careful context

>* Means compare current and target performance
>* Use averages with variation and risk context



In [ ]:
#@title Python Code - Mean Metrics

# Mean metrics summarize typical engineering performance.
# Small datasets keep calculations easy to inspect.
# Averages need context from unusual observations.

# Import numerical tools for simple calculations.
import numpy as np

# Store hourly output from one production line.
units_per_hour = np.array([118, 121, 119, 123, 117, 120])

# Store cycle times for repeated assembly tasks.
cycle_seconds = np.array([42.5, 41.8, 43.1, 42.2, 60.4, 42.0])

# Validate both datasets contain observations.
assert units_per_hour.size > 0 and cycle_seconds.size > 0

# Compute arithmetic means for both metrics.
mean_units = units_per_hour.mean()
mean_cycle = cycle_seconds.mean()

# Compare with a second cycle-time mean.
typical_cycle = cycle_seconds[cycle_seconds < 50].mean()

# Print compact results for engineering interpretation.
print(f"Average output: {mean_units:.1f} units per hour")
print(f"Average cycle time: {mean_cycle:.1f} seconds")
print(f"Typical cycle time excluding stoppage: {typical_cycle:.1f} seconds")
print("Interpretation: the stoppage raises the cycle-time mean.")



### **1.2. Range**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Industrial Engineers/Module_04/Lecture_A/image_01_02.jpg?v=1780942187" width="250">



>* Range shows spread between smallest and largest values
>* Large ranges may signal unstable processes

>* Range quickly compares process consistency.
>* Wide ranges signal possible downstream problems.

>* Range is sensitive to extreme values
>* Use range to investigate process variation



In [ ]:
#@title Python Code - Range

# Range measures spread between process extremes.
# Engineers use range for quick checks.
# This example studies machining cycle times.

cycle_times_minutes = [4.8, 5.1, 4.9, 5.0, 6.2, 4.7]

# Validate that enough observations exist.
if len(cycle_times_minutes) < 2:
    raise ValueError("Need at least two observations.")

# Find smallest and largest observed times.
minimum_time = min(cycle_times_minutes)
maximum_time = max(cycle_times_minutes)
cycle_time_range = maximum_time - minimum_time

# Compute a simple average for context.
average_time = sum(cycle_times_minutes) / len(cycle_times_minutes)

# Compare the range with a practical threshold.
threshold_minutes = 1.0
needs_review = cycle_time_range > threshold_minutes

# Print compact descriptive results.
print("Cycle times, minutes:", cycle_times_minutes)
print("Minimum time:", minimum_time)
print("Maximum time:", maximum_time)
print("Range:", round(cycle_time_range, 2), "minutes")

# Interpret the range for improvement decisions.
print("Average time:", round(average_time, 2), "minutes")
print("Review needed:", needs_review)
print("Decision: investigate variability" if needs_review else "Decision: monitor process")



### **1.3. Frequency Distributions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Industrial Engineers/Module_04/Lecture_A/image_01_03.jpg?v=1780942189" width="250">



>* Group process data to reveal patterns
>* Use intervals to interpret process behavior

>* Show clustering, spread, and unusual patterns
>* Guide decisions beyond average values

>* Notice distribution shape and extreme values
>* Match improvement actions to observed patterns



In [ ]:
#@title Python Code - Frequency Distributions

# Frequency distributions summarize repeated process measurements.
# Engineers use bins to reveal performance patterns.
# This example groups assembly cycle times.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a small deterministic process dataset.
rng = np.random.default_rng(seed=42)
cycle_times = rng.normal(loc=5.2, scale=0.45, size=40)

# Add a few longer cycles representing delays.
delays = np.array([6.4, 6.7, 7.1, 7.4])
cycle_times = np.concatenate([cycle_times, delays])

# Validate the dataset before summarizing.
assert cycle_times.size > 0, "Cycle time data are empty"
assert np.all(cycle_times > 0), "Cycle times must be positive"

# Define engineering-friendly time intervals.
bin_edges = [4.0, 4.8, 5.6, 6.4, 7.2, 8.0]
bin_labels = ["Very fast", "Typical", "Slightly slow", "Delayed", "Severe"]

# Count observations falling in each interval.
category_data = pd.cut(cycle_times, bins=bin_edges, labels=bin_labels)
frequency_table = category_data.value_counts(sort=False)

# Convert counts into percentages for interpretation.
percent_table = frequency_table / frequency_table.sum() * 100
summary_table = pd.DataFrame({"Count": frequency_table, "Percent": percent_table.round(1)})

# Print a compact frequency distribution table.
print("Cycle time frequency distribution, in minutes:")
print(summary_table.to_string())

# Highlight one improvement interpretation.
slow_share = percent_table.loc[["Delayed", "Severe"]].sum()
print(f"Delayed or severe cycles: {slow_share:.1f}% of observations")

# Plot exactly one chart for visual interpretation.
plt.figure(figsize=(7, 4))
plt.bar(summary_table.index.astype(str), summary_table["Count"], color="steelblue")

# Label the frequency distribution clearly.
plt.title("Assembly Cycle Time Frequency Distribution")
plt.xlabel("Cycle time category")
plt.ylabel("Number of units")

# Improve readability before showing the chart.
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()



## **2. Operational Performance Metrics**

### **2.1. Utilization Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Industrial Engineers/Module_04/Lecture_A/image_02_01.jpg?v=1780942191" width="250">



>* Utilization compares working time to available time
>* Python converts time records into performance indicators

>* Calculate utilization from structured operational data
>* Use Python for grouped, repeatable analysis

>* Balance low and high utilization risks
>* Use context to guide improvement decisions



In [ ]:
#@title Python Code - Utilization Metrics

# This example calculates machine utilization metrics.
# Small production data keeps results understandable.
# Managers use utilization to balance capacity.

import pandas as pd
import matplotlib.pyplot as plt

# Create simple shift records for three machines.
data = {
    "machine": ["Mill A", "Lathe B", "Press C"],
    "scheduled_hours": [8.0, 8.0, 8.0],

    "active_hours": [6.2, 7.4, 4.9],
    "downtime_hours": [0.8, 0.3, 1.6],
}

# Convert the dictionary into a table.
df = pd.DataFrame(data)

# Validate that all scheduled hours are positive.
if (df["scheduled_hours"] <= 0).any():
    raise ValueError("Scheduled hours must be positive")

# Calculate utilization and idle time.
df["utilization_pct"] = (
    df["active_hours"] / df["scheduled_hours"] * 100
)

df["idle_hours"] = (
    df["scheduled_hours"] - df["active_hours"] - df["downtime_hours"]
)

# Select concise columns for display.
summary = df[
    ["machine", "utilization_pct", "downtime_hours", "idle_hours"]
]

# Print rounded results without overwhelming output.
print("Utilization summary by machine:")
print(summary.round(1).to_string(index=False))

# Identify the lowest utilization resource.
lowest_row = df.loc[df["utilization_pct"].idxmin()]
message = f"Improvement focus: {lowest_row['machine']} has lowest utilization."
print(message)

# Plot one clear utilization comparison chart.
plt.figure(figsize=(6, 4))
plt.bar(df["machine"], df["utilization_pct"], color="steelblue")

plt.axhline(85, color="orange", linestyle="--", label="Target 85%")
plt.ylabel("Utilization percent")
plt.title("Machine Utilization During One Shift")
plt.ylim(0, 100)

plt.legend()
plt.tight_layout()
plt.show()



### **2.2. Throughput Calculation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Industrial Engineers/Module_04/Lecture_A/image_02_02.jpg?v=1780942193" width="250">



>* Measure completed output per time period
>* Use Python to group and filter records

>* Define time, output, and process boundaries
>* Use Python filters, timestamps, and grouping

>* Compare throughput with related process metrics
>* Use Python to guide improvement decisions



In [ ]:
#@title Python Code - Throughput Calculation

# This example calculates hourly process throughput.
# Completed units define useful operational output.
# Small data keeps the lecture example clear.

import pandas as pd
import matplotlib.pyplot as plt

# Create small production records for one shift.
records = [
    {"hour": "08:00", "line": "A", "completed": 42},
    {"hour": "09:00", "line": "A", "completed": 48},

    {"hour": "10:00", "line": "A", "completed": 39},
    {"hour": "11:00", "line": "A", "completed": 51},
    {"hour": "08:00", "line": "B", "completed": 36},

    {"hour": "09:00", "line": "B", "completed": 40},
    {"hour": "10:00", "line": "B", "completed": 44},
    {"hour": "11:00", "line": "B", "completed": 38},
]

# Convert records into a compact table.
data = pd.DataFrame(records)
assert len(data) == 8, "Expected eight production records."

# Sum completed units by production line.
line_output = data.groupby("line")["completed"].sum()
observed_hours = data["hour"].nunique()

# Calculate throughput as completed units per hour.
line_throughput = line_output / observed_hours
best_line = line_throughput.idxmax()

# Print only key teaching results.
print("Throughput by line, completed units per hour:")
print(line_throughput.round(1).to_string())
print(f"Highest throughput line: {best_line}")

# Prepare hourly totals for one simple plot.
hourly_output = data.groupby("hour")["completed"].sum()
ax = hourly_output.plot(kind="bar", color="steelblue")

# Label the throughput chart clearly.
ax.set_title("Hourly Throughput Across Both Lines")
ax.set_xlabel("Shift Hour")
ax.set_ylabel("Completed Units")
plt.tight_layout()

plt.show()



### **2.3. Yield Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Industrial Engineers/Module_04/Lecture_A/image_02_03.jpg?v=1780942195" width="250">



>* Yield measures acceptable first-pass output
>* Python turns records into process insight

>* Define good output and quality losses
>* Group yield data to locate problems

>* Compare yield with operational context
>* Track trends to guide improvements



In [ ]:
#@title Python Code - Yield Metrics

# Yield metrics quantify acceptable production output.
# Small datasets make calculations easier to inspect.
# Engineers compare yield across operating conditions.

# Import pandas for tabular process records.
import pandas as pd

# Create small inspection records by shift.
records = [
    {"shift": "Day", "started": 500, "passed": 475, "reworked": 15, "scrap": 10},
    {"shift": "Evening", "started": 480, "passed": 440, "reworked": 25, "scrap": 15},
    {"shift": "Night", "started": 450, "passed": 405, "reworked": 30, "scrap": 15},
]

# Convert records into a dataframe.
df = pd.DataFrame(records)

# Confirm required count columns are present.
required_cols = {"started", "passed", "reworked", "scrap"}
assert required_cols.issubset(df.columns)

# Check all shift sizes are positive.
assert (df["started"] > 0).all()

# Calculate first pass yield percentage.
df["first_pass_yield"] = df["passed"] / df["started"]

# Calculate final yield after rework succeeds.
df["final_yield"] = (df["passed"] + df["reworked"]) / df["started"]

# Calculate scrap rate for quality loss.
df["scrap_rate"] = df["scrap"] / df["started"]

# Compute overall totals across shifts.
totals = df[["started", "passed", "reworked", "scrap"]].sum()

# Compute overall first pass yield.
overall_fpy = totals["passed"] / totals["started"]

# Compute overall final yield.
overall_final = (totals["passed"] + totals["reworked"]) / totals["started"]

# Print compact shift results.
print("Yield by shift:")

# Show only essential rounded metrics.
summary = df[["shift", "first_pass_yield", "final_yield", "scrap_rate"]].copy()

# Format metrics as percentages.
for col in ["first_pass_yield", "final_yield", "scrap_rate"]:
    summary[col] = (summary[col] * 100).round(1).astype(str) + "%"

# Print the small summary table.
print(summary.to_string(index=False))

# Print overall process performance.
print(f"Overall first pass yield: {overall_fpy:.1%}")

# Interpret improvement priority simply.
worst_shift = df.loc[df["first_pass_yield"].idxmin(), "shift"]

# Print decision-oriented interpretation.
print(f"Investigate the {worst_shift} shift for hidden quality losses.")

# Print final yield after rework.
print(f"Overall final yield after rework: {overall_final:.1%}")



## **3. Process Improvement Signals**

### **3.1. Bottleneck Signals**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Industrial Engineers/Module_04/Lecture_A/image_03_01.jpg?v=1780942182" width="250">



>* Bottlenecks limit total process performance
>* Queues reveal where improvements matter most

>* Confirm bottlenecks using multiple flow metrics
>* Improve constraints, inflow, and priority rules

>* Improve bottlenecks for system-wide performance gains
>* Use flow patterns to target lasting interventions



In [ ]:
#@title Python Code - Bottleneck Signals

# Bottleneck signals compare flow metrics.
# Small tables can reveal constraints.
# Improvement should target system limits.

import pandas as pd
import matplotlib.pyplot as plt

# Create compact process performance data.
stations = ["Filling", "Labeling", "Boxing"]
cycle_minutes = [0.9, 1.4, 0.8]
queue_jobs = [12, 48, 5]
utilization = [0.72, 0.96, 0.61]

# Combine metrics into one table.
data = pd.DataFrame({
    "station": stations,
    "cycle_minutes": cycle_minutes,
    "queue_jobs": queue_jobs,
    "utilization": utilization,
})

# Estimate hourly capacity for each station.
data["capacity_per_hour"] = 60 / data["cycle_minutes"]
system_capacity = data["capacity_per_hour"].min()

# Identify the strongest bottleneck signal.
data["signal_score"] = data["utilization"] * data["queue_jobs"]
bottleneck_row = data.loc[data["signal_score"].idxmax()]

# Print only key decision information.
print("Process improvement bottleneck signal")
print(f"System capacity: {system_capacity:.1f} units per hour")
print(f"Likely bottleneck: {bottleneck_row['station']}")
print(f"Utilization there: {bottleneck_row['utilization']:.0%}")
print(f"Queue before it: {bottleneck_row['queue_jobs']} jobs")
print("Decision: improve this step before nonconstraints.")

# Plot queue and utilization together.
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.bar(data["station"], data["queue_jobs"], color="lightsteelblue")
ax1.set_ylabel("Queue before station, jobs")

# Add utilization as a second signal.
ax2 = ax1.twinx()
ax2.plot(data["station"], data["utilization"], color="crimson", marker="o")
ax2.set_ylabel("Utilization fraction")

# Highlight interpretation in the title.
plt.title("Bottleneck Signal: High Queue Plus High Utilization")
fig.tight_layout()
plt.show()



### **3.2. Downtime Decision Signals**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Industrial Engineers/Module_04/Lecture_A/image_03_02.jpg?v=1780942180" width="250">



>* Downtime reveals process weaknesses and support gaps
>* Patterns guide targeted improvement actions

>* Separate downtime symptoms from true causes
>* Use context to judge improvement urgency

>* Prioritize downtime with the greatest operational impact
>* Choose fixes that improve future process reliability



In [ ]:
#@title Python Code - Downtime Decision Signals

# Downtime patterns guide practical improvement decisions.
# Small datasets can reveal operational priorities.
# Metrics become useful when interpreted contextually.

import pandas as pd
import matplotlib.pyplot as plt

# Create a small shift summary dataset.
records = [
    {"cause": "Sealer failure", "events": 1, "minutes": 120},
    {"cause": "Label waiting", "events": 12, "minutes": 60},
    {"cause": "Changeover delay", "events": 3, "minutes": 45},
    {"cause": "Inspection hold", "events": 2, "minutes": 30},
]

# Convert records into a table.
downtime = pd.DataFrame(records)
shift_minutes = 480

# Validate the small teaching dataset.
assert len(downtime) > 0
assert downtime["minutes"].sum() <= shift_minutes

# Calculate decision oriented downtime metrics.
downtime["share"] = downtime["minutes"] / downtime["minutes"].sum()
downtime["average_event"] = downtime["minutes"] / downtime["events"]
downtime["signal"] = downtime["average_event"].apply(lambda value: "long" if value >= 40 else "recurring")

# Sort causes by total lost minutes.
downtime = downtime.sort_values("minutes", ascending=False)
total_down = downtime["minutes"].sum()
down_rate = total_down / shift_minutes

# Print concise interpretation lines.
print(f"Total downtime: {total_down} of {shift_minutes} minutes.")
print(f"Downtime rate: {down_rate:.1%} of scheduled time.")
print("Top downtime causes and decision signals:")

# Report only compact priority signals.
for row in downtime.itertuples():
    cause_text = row.cause
    message = f"{cause_text}: {row.minutes} min, {row.signal} signal."
    print(message)

# Identify the most urgent improvement candidate.
priority = downtime.iloc[0]
action = "maintenance focus" if priority.signal == "long" else "coordination focus"
print(f"Priority action: {action} for {priority.cause}.")

# Plot one simple Pareto style bar chart.
plt.figure(figsize=(7, 4))
plt.bar(downtime["cause"], downtime["minutes"], color="steelblue")
plt.title("Downtime Decision Signals by Cause")
plt.ylabel("Downtime minutes")

# Improve label readability for beginners.
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()



### **3.3. Sustained Improvement Signals**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Industrial Engineers/Module_04/Lecture_A/image_03_03.jpg?v=1780942183" width="250">



>* Repeated metrics show lasting process improvement
>* Evidence guides standardization or further investigation

>* Check metrics improve together sustainably
>* Confirm gains remain reliable over time

>* Verify improvements across teams and disruptions
>* Use sustained evidence for implementation decisions



In [ ]:
#@title Python Code - Sustained Improvement Signals

# Sustained signals need repeated performance evidence.
# Small datasets can reveal stable improvement.
# Metrics guide practical engineering decisions.

import pandas as pd
import matplotlib.pyplot as plt

# Create compact shift performance records.
data = {
    "day": [1, 2, 3, 4, 5, 6],
    "throughput": [480, 505, 510, 512, 515, 517],
    "yield_rate": [0.925, 0.941, 0.944, 0.946, 0.947, 0.948],
    "downtime_minutes": [58, 44, 42, 40, 39, 38],
}

# Store records in a small dataframe.
df = pd.DataFrame(data)

# Confirm enough observations for sustained signals.
if len(df) < 4:
    raise ValueError("Need at least four observations.")

# Compare early baseline with recent performance.
baseline = df.iloc[:2].mean(numeric_only=True)
recent = df.iloc[-3:].mean(numeric_only=True)

# Calculate practical metric changes.
throughput_gain = recent["throughput"] - baseline["throughput"]
yield_gain = recent["yield_rate"] - baseline["yield_rate"]
downtime_drop = baseline["downtime_minutes"] - recent["downtime_minutes"]

# Check whether recent days remain consistently better.
recent_rows = df.tail(3)
stable_throughput = recent_rows["throughput"].min() > baseline["throughput"]
stable_yield = recent_rows["yield_rate"].min() > baseline["yield_rate"]
stable_downtime = recent_rows["downtime_minutes"].max() < baseline["downtime_minutes"]

# Combine signals into one improvement decision.
sustained_signal = all([
    stable_throughput,
    stable_yield,
    stable_downtime,
])

# Print concise interpretation for engineers.
print("Baseline throughput:", round(baseline["throughput"], 1), "units/day")
print("Recent throughput gain:", round(throughput_gain, 1), "units/day")
print("Recent yield gain:", round(yield_gain * 100, 2), "percentage points")
print("Recent downtime drop:", round(downtime_drop, 1), "minutes/day")
print("Sustained improvement signal:", sustained_signal)

# Show how metrics moved across time.
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(df["day"], df["throughput"], marker="o", label="Throughput")
ax1.set_xlabel("Production day")
ax1.set_ylabel("Units per day")

# Add downtime on a second axis.
ax2 = ax1.twinx()
ax2.plot(df["day"], df["downtime_minutes"], color="red", marker="s", label="Downtime")
ax2.set_ylabel("Downtime minutes")

# Finish with readable plot formatting.
plt.title("Sustained Improvement Signals")
fig.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Performance Metrics**</font>


In this lecture, you learned to:
- Compute descriptive measures for production and process performance data. 
- Calculate utilization, throughput, yield, and downtime metrics in Python. 
- Interpret metric results in relation to process improvement decisions. 

In the next Lecture (Lecture B), we will go over 'Inventory and Queues'